## Gradient Check (Theory)


### Why Gradient Check is Needed?

우리는 역전파로 ```dW2 = A1.T @ dZ2```의 값을 계산했다.

이때, shape이 맞고 loss가 감소하더라도, 이 식이 정확하다고 단정할 수는 없다. 틀린 gradient라도 우연히 loss를 감소시킬 수 있기 때문이다.

그래서 서로 다른 두 방법으로 gradient를 계산해 비교한다.

* Analytical gradient: 미분식과 역전파로 계산

* Numerical gradient: 파라미터를 아주 조금 움직여 loss 변화를 관찰

<br>

### Numerical Gradient의 직관

함수 $$f(x) = x^2$$ 를 가정하자

이것을 미분하면? $$f'(x) = 2x $$

따라서 실제 $$f'(3) = 6 $$

하지만, 이것을 모른다고 가정하자.

그렇다면 고등학교에서 배웠듯이 x를 아주 미세하게, 오른쪽과 왼쪽으로 움직여보자

$$\frac{f(3+h)-f(3-h)}{2h}$$

$h=0.001$이라면:

$$
f(3.001)=3.001^2
$$

$$
f(2.999)=2.999^2
$$

이 두 함수값의 차이를 전체 이동 거리인 $0.002$로 나누면 약 6이 나온다. 이것이 numerical gradient이다.

이때 일반적인 기울기를 구하는 방식 대신, 양쪽 값 모두를 쓸까?

<br>

### Forward Divided Difference vs Central Difference


1. Forward Divided Difference (전진 차분)

    우리가 흔히 아는 그거. (식은 적기 귀찮으니 패스함)

    약간의 오차가 생긴다


2. Central Difference (중앙 차분)


    $x-h$에서 $x+h$까지, $x$를 중심으로 대칭인 구간의 평균 기울기를 사용한다

    실제 기울기와 일치하게 된다.

<br>

### 왜 중앙 차분이 더 정확할까?

$f(x)=x^2$의 그래프는 곡선이다

$x$와 $x+h$ 사이의 기울기는 $x$보다 약간 오른쪽의 기울기를 반영한다

$x-h$와 $x$ 사이의 기울기는 $x$보다 약간 왼쪽의 기울기를 반영한다

두 방향을 함께 사용하면 왼쪽 오차와 오른쪽 오차가 서로 상쇄된다.

전진 차분은 오른쪽으로 치우친 측정이고, 중앙 차분은 우리가 알고 싶은 $x$를 가운데에 놓은 대칭적인 측정이 된다

수학적으로도 전진 차분의 오차는 대략 $O(h)$, 중앙 차분의 오차는 $O(h^2)$이다.

예를 들어 $h=0.01$이라면,

- 전진 차분의 대표적인 오차 규모: 약 $0.01$
- 중앙 차분의 대표적인 오차 규모: 약 $0.0001$

그래서 역전파가 정확한지 엄격하게 검사하는 **gradient check**에는 중앙 차분을 사용한다.

## What Should We Adjust in a Neural Network?

신경망의 Loss는 모든 파라미터에 의해 결정된다.

$$L = L(W_1, b_1, W_2, b_2)$$

하지만, numerical gradient를 구할 때는 파라미터 전체를 한 번에 움직이지 않는다

**원소 하나만 움직이고, 나머지는 모두 고정한다**

예를 들어 `W2[0, 0]`의 gradient를 확인한다면,

$$
\frac{\partial L}{\partial W_2[0,0]}
\approx
\frac{
L(W_2[0,0]+h)-L(W_2[0,0]-h)
}{2h}
$$

계산 방법

1. `W2[0, 0]`에 $h$를 더한다.

2. 순전파를 다시 수행해 `loss_plus`를 구한다.

3. `W2[0, 0]`에 $h$를 뺀다.

4. 순전파를 다시 수행해 `loss_minus`를 구한다.

5. 중앙 차분을 계산한다.

6. 원래 `W2[0, 0]` 값으로 복구한다.

이때, Numerical gradient를 구할 때는 파라미터 업데이트나 학습을 하면 안 된다




### 음수 Gradient인데, 왜 W는 증가할까?

가중치는 다음과 같이 업데이트된다

$$
w_{\text{new}}
=
w
-
\eta
\frac{\partial L}{\partial w}
$$

여기서,

- $w$ : 현재 가중치

- $w_{\text{new}}$ : 업데이트된 가중치

- $\eta$ : 학습률(Learning Rate)

- $\frac{\partial L}{\partial w}$ : 손실 함수에 대한 가중치의 gradient



$$w_{\text{new}}=w-\eta\frac{\partial L}{\partial w}$$

gradient가 -8, 학습률이 0.1 이라면:

$$
w_{\text{new}}
=1-0.1(-8)
=1+0.8
=1.8
$$ 

따라서 w가 증가한다

gradient의 부호는 loss가 증가하는 방향을 알려준다


* 양수 gradient: $w$를 증가시키면 loss가 증가

* 음수 gradient: $w$를 증가시키면 loss가 감소

* SGD: gradient의 반대 방향으로 이동

현재는 예측값이 정답보다 작기 때문에 $w$를 증가시켜야 한다 

numerical gradient가 정확히 그 방향을 알려준 것이다

### Analytical gradient와 비교

손실 함수를

$$
L=(xw-y)^2
$$

라고 하자

편의를 위해

$$
\hat{y}=xw
$$

라고 하면 손실 함수는

$$
L=(\hat{y}-y)^2
$$

가 된다

Chain Rule에 따라

$$
\frac{\partial L}{\partial w}
=
\frac{\partial L}{\partial \hat{y}}
\frac{\partial \hat{y}}{\partial w}
$$

각각 계산하면,

$$
\frac{\partial L}{\partial \hat{y}}
=
2(\hat{y}-y)
$$

$$
\frac{\partial \hat{y}}{\partial w}
=
x
$$

따라서,

$$
\frac{\partial L}{\partial w}
=
2(\hat{y}-y)x
$$

현재 값

$$
x=2,\qquad y=4,\qquad w=1
$$

을 대입하면,

$$
\hat{y}=xw=2\times1=2
$$

따라서,

$$
\frac{\partial L}{\partial w}
=
2(2-4)\times2
=
-8
$$

따라서 같아진다.

두 값이 일치하므로 analytical gradient 구현이 정확하다고 판단할 수 있다. 이것이 gradient check의 핵심이 된다

### 오차를 어떻게 비교할까?

실제 신경망에서는 부동소수점 계산 때문에 완전히 같지는 않을 수도 있다.

analytical = -7.9999998
numerical  = -8.0000001

따라서, 단순히 ```==```이 아니라, 상대오차를 사용한다

$$
\text{relative error}
=
\frac{|g_a-g_n|}
{\max\left(10^{-8},\,|g_a|+|g_n|\right)}
$$




## 실제로 MLP에서의 적용 (Code)

analytical_gradient = dW2[0, 0]
numerical_gradient = 중앙 차분으로 계산한 값

```X → W1 → ReLU → W2 → Sigmoid → Loss```의 구조를 갖는다고 하자.

In [2]:
import numpy as np

rng = np.random.default_rng(42)

X = np.array([
    [0.0, 0.0],
    [0.0, 1.0],
    [1.0, 0.0],
    [1.0, 1.0],
])

y = np.array([
    [0.0],
    [1.0],
    [1.0],
    [0.0],
])

W1 = rng.standard_normal((2, 8)) * 0.1
b1 = np.zeros((1, 8))

W2 = rng.standard_normal((8, 1)) * 0.1
b2 = np.zeros((1, 1))

In [3]:
print("X:", X.shape)
print("y:", y.shape)
print("W1:", W1.shape)
print("b1:", b1.shape)
print("W2:", W2.shape)
print("b2:", b2.shape)

X: (4, 2)
y: (4, 1)
W1: (2, 8)
b1: (1, 8)
W2: (8, 1)
b2: (1, 1)


In [4]:
Z1 = X @ W1 + b1

# ReLU
A1 = np.maximum(0, Z1)

Z2 = A1 @ W2 + b2

# Sigmoid
Y_hat = 1 / (1 + np.exp(-Z2))

In [5]:
epsilon = 1e-12

loss = -np.mean(
    y * np.log(Y_hat + epsilon)
    + (1 - y) * np.log(1 - Y_hat + epsilon)
)

print("Z1:", Z1.shape)
print("A1:", A1.shape)
print("Z2:", Z2.shape)
print("Y_hat:", Y_hat.shape)
print("loss:", loss)

Z1: (4, 8)
A1: (4, 8)
Z2: (4, 1)
Y_hat: (4, 1)
loss: 0.694132297612279


In [6]:
N = X.shape[0]

dZ2 = (Y_hat - y) / N
dW2 = A1.T @ dZ2

analytical_gradient = dW2[0, 0]

print("dZ2 shape:", dZ2.shape)
print("dW2 shape:", dW2.shape)
print("analytical gradient:", analytical_gradient)

dZ2 shape: (4, 1)
dW2 shape: (8, 1)
analytical gradient: -0.00015401115034865184


In [7]:
# Numerical gradient

original = W2[0, 0]
h = 1e-5

W2[0, 0] = original + h

Z1_plus = X @ W1 + b1
A1_plus = np.maximum(0, Z1_plus)
Z2_plus = A1_plus @ W2 + b2
Y_hat_plus = 1 / (1 + np.exp(-Z2_plus))

loss_plus = -np.mean(
    y * np.log(Y_hat_plus + epsilon)
    + (1 - y) * np.log(1 - Y_hat_plus + epsilon)
)

W2[0, 0] = original

print("original:", original)
print("loss:", loss)
print("loss_plus:", loss_plus)
print("restored:", W2[0, 0])

original: 0.036875078408249884
loss: 0.694132297612279
loss_plus: 0.6941322960721731
restored: 0.036875078408249884


In [8]:
W2[0, 0] = original - h

Z1_minus = X @ W1 + b1
A1_minus = np.maximum(0, Z1_minus)
Z2_minus = A1_minus @ W2 + b2
Y_hat_minus = 1 / (1 + np.exp(-Z2_minus))

loss_minus = -np.mean(
    y * np.log(Y_hat_minus + epsilon)
    + (1 - y) * np.log(1 - Y_hat_minus + epsilon)
)

# 검사 후 원래 값으로 복구
W2[0, 0] = original

In [9]:
numerical_gradient = (loss_plus - loss_minus) / (2 * h)

relative_error = (
    abs(analytical_gradient - numerical_gradient)
    / max(
        1e-8,
        abs(analytical_gradient) + abs(numerical_gradient)
    )
)

# numerical_gradient를 구하고, analytical gradient와 비교해서 상대 오차를 구해준다

In [10]:
print("loss_plus:", loss_plus)
print("loss_minus:", loss_minus)
print("analytical gradient:", analytical_gradient)
print("numerical gradient:", numerical_gradient)
print("relative error:", relative_error)
print("restored:", W2[0, 0])

loss_plus: 0.6941322960721731
loss_minus: 0.6941322991523962
analytical gradient: -0.00015401115034865184
numerical gradient: -0.00015401115383006925
relative error: 1.1302484800839616e-08
restored: 0.036875078408249884


### Gradient Checking

$$
\text{relative error}
=
\frac{\left|g_a-g_n\right|}
{\max\left(10^{-8},\,\left|g_a\right|+\left|g_n\right|\right)}
$$

- **relative error < $10^{-7}$** : 거의 완벽 (역전파 구현이 매우 정확)

- **$10^{-7} \le$ relative error $< 10^{-5}$** : 매우 좋음 (일반적으로 문제 없음)

- **$10^{-5} \le$ relative error $< 10^{-3}$** : 구현을 다시 확인해볼 필요가 있음

- **relative error $\ge 10^{-3}$** : 버그가 있을 가능성이 큼

수치 미분도 근사값이다.

컴퓨터는 실수를 무한하게 정확하게 표현하지 못하기에 상대 오차가 정확히 0이 되는 경우는 매우 드물다

In [11]:
def compute_loss():
    # 순전파
    Z1 = X @ W1 + b1 # z = wx + b
    A1 = np.maximum(0, Z1) # ReLU

    Z2 = A1 @ W2 + b2
    Y_hat = 1 / (1 + np.exp(-Z2)) # Sigmoid

    return -np.mean(
        y * np.log(Y_hat + epsilon)
        + (1 - y) * np.log(1 - Y_hat + epsilon)
    ) # Loss

In [12]:
# W2의 모든 원소에 대해 Numerical Gradient를 계산한다

numerical_dW2 = np.zeros_like(W2)
# np.zeros_like(): 기존 배열과 똑같은 shape을 가지는 0으로 채워진 배열 만들기, 주로 gradient를 저장할 배열 만들때 사용

for index in np.ndindex(W2.shape):
    # np.ndindex(): N차원 배열의 모든 인덱스를 하나씩 순회하는 함수
    
    original = W2[index]

    W2[index] = original + h
    loss_plus = compute_loss()

    W2[index] = original - h
    loss_minus = compute_loss()

    W2[index] = original

    numerical_dW2[index] = (
        loss_plus - loss_minus
    ) / (2 * h)

# np.ndindex(W2.shape)는 다음 인덱스를 차례대로 만들어준다, 그렇기에 모든 인덱스에 대해서 다 돌 수 있음

In [13]:
# 전체 원소의 상대 오차 계산

elementwise_error = (
    np.abs(dW2 - numerical_dW2)
    / np.maximum(
        1e-8,
        np.abs(dW2) + np.abs(numerical_dW2)
    )
)

print("analytical dW2:")
print(dW2)

print("\nnumerical dW2:")
print(numerical_dW2)

print("\nmaximum relative error:")
print(np.max(elementwise_error))

analytical dW2:
[[-0.00015401]
 [ 0.        ]
 [ 0.00029225]
 [ 0.00031144]
 [-0.00082322]
 [-0.0140535 ]
 [ 0.0001035 ]
 [ 0.        ]]

numerical dW2:
[[-0.00015401]
 [ 0.        ]
 [ 0.00029225]
 [ 0.00031144]
 [-0.00082322]
 [-0.0140535 ]
 [ 0.0001035 ]
 [ 0.        ]]

maximum relative error:
1.246206533442286e-08


```python
dW2[1,0] = 0
dW2[7,0] = 0
```

이처럼 ```dw2```에 0이 있을 수도 있다.
이는 오류가 아니라, $$dW_2 = A_1^TdZ_2$$ 계산 방법에 따라, 특정 은닉 뉴런의 ReLU 출력이 데이터 4개 모두에서 0이라면, 그 뉴런과 출력층을 연결하는 가중치는 현재 loss에 영향을 주지 않는다.

따라서 해당 W2 원소의 gradient도 0이 된다.

In [14]:
# 출력층 bias의 gradient 계산

db2 = np.sum(dZ2, axis=0, keepdims=True)
# bias b2는 모든 데이터에 동일하게 더해지기 때문에, 편미분과 Chain Rule에 따라 유도할 수 있다.


# 은닉층 방향으로 gradient를 전달한다

dA1 = dZ2 @ W2.T
dZ1 = dA1 * (Z1 > 0)
# A1은 ReLU를 통과한 값이기에, ReLU의 도함수를 곱해준다


# 첫 번째 계층의 gradient 계산

dW1 = X.T @ dZ1
db1 = np.sum(dZ1, axis=0, keepdims=True)

# Chain Rule로 얻은 각 원소의 미분을, 배치와 뉴런 전체에 대해 효율적으로 계산하고 합산한 형태가 곧 행렬곱임

print("dW1:", dW1.shape)
print("db1:", db1.shape)
print("dW2:", dW2.shape)
print("db2:", db2.shape)

dW1: (2, 8)
db1: (1, 8)
dW2: (8, 1)
db2: (1, 1)


### 중간점검

내가 확실히 아는 것

1. gradient check

2. 왜 중앙차분을 쓰는지

3. SGD 방향

4. 파라미터 복구


내가 헷갈리는 것

1. ReLU와 ```dW2 = 0```

2. Shape 맞추기 - 얘는 해도 해도 개빡치게 계속 헷갈리네

### 중간점검에 대한 보완 - ReLU와 ```dW2 = 0```

forward propagation

$$A_1=\operatorname{ReLU}(Z_1)\$$

backpropagation

$$dZ_1=dA_1\cdot\mathbf{1}(Z_1>0)\$$

```dW2```는 ReLU의 도함수를 직접 사용하지 않았다

$$dW_2=A_1^T dZ_2$$

다만 특정 은닉층 뉴런 ```A1```값이 모든 Data에서 0이라면?

그 뉴런과 연결된 ```W2[j,0]```의 gradient는:
$$
dW_2[j,0]
=
A_1[:,j]^TdZ_2
=
0
$$ 

이 된다.

즉, 정리하면
1. ReLU가 해당 뉴런의 A1을 0으로 만듦

2. dW2는 A1을 사용함

3. 따라서 해당 dW2가 0이 됨

반면 ReLU의 도함수는 dZ1을 계산할 때 직접 사용된다

### 중간점검에 대한 보완 - Shape

가장 중요한 규칙은, row는 데이터 개수, Column은 해당 층의 뉴런 개수(즉 특성, 성질)

예시를 들어보자
```
X:   (4, 2)
W1:  (2, 8)
b1:  (1, 8)
W2:  (8, 1)
b2:  (1, 1)
```


$$Z_1=XW_1+b_1$$

공식에 따라서,

```
X @ W1
(4,2) @ (2,8)
```

```Z1: (4,8)```

즉, 데이터는 4개 각 뉴런에 대한 은닉층 뉴런의 출력이 8개가 된다는 말이다.

```dZ2```는?

Z2의 shape부터 보면:
A1 @ W2
(4,8) @ (8,1) → (4,1)

dZ2는 Z2 각각에 대한 gradient이므로 동일한 shape가 된다

dZ2: (4,1)

아 이제 알겠다...

내가 놓치고 있었던 곳: Z를 그냥 W랑 비슷하게 보았음.. W는 약간 뉴런과 뉴런을 연결하는 연결고리 (가중치 행렬이니까)

그리고 Z는 그 계산을 행한 후니까



### 반복문을 함수로 일반화

다시 돌아와서, 이전에는 dW2만 검사하도록 했다. ```numerical_dW2 = np.zeros_like(W2)```

하지만, numerical gradient의 원리는 W1, b1, W2, b2 모두 동일하다

즉

1. 원소 하나 선택

2. original + h의 loss 계산

3. original - h의 loss 계산

4. 원래 값 복구

5. 중앙 차분 저장

따라서 검사할 array를 ```parameter```라는 인수로 받아서 검사할 수 있도록 할 수 있다.

In [15]:
def numerical_gradient(parameter, h=1e-5):
    # parameter은 실제 배열을 가리킨다
    gradient = np.zeros_like(parameter)

    for index in np.ndindex(parameter.shape):
        original = parameter[index]
        # 실제 값을 변경하기에, 복구 코드는 필수!

        parameter[index] = original + h
        loss_plus = compute_loss()

        parameter[index] = original - h
        loss_minus = compute_loss()

        parameter[index] = original

        gradient[index] = (
            loss_plus - loss_minus
        ) / (2 * h)

    return gradient

In [16]:
numerical_dW2_function = numerical_gradient(W2)

print(numerical_dW2_function)
print(
    "maximum difference:",
    np.max(np.abs(numerical_dW2_function - numerical_dW2))
)

# numerical_dW2: 이전에 직접 작성한 W2 반복문 결과
# numerical_dW2_function: 새 함수 결과

[[-0.00015401]
 [ 0.        ]
 [ 0.00029225]
 [ 0.00031144]
 [-0.00082322]
 [-0.0140535 ]
 [ 0.0001035 ]
 [ 0.        ]]
maximum difference: 0.0


### 최대 상대오차 함수

먼저, analytical gradient와 numerical gradient의 최대 상대오차를 반환하는 함수를 작성한다

In [17]:
def max_relative_error(analytical, numerical):
    # 입력으로 역전파로 계산한 gradient와 수치 미분으로 계산한 gradient가 들어온다

    elementwise_error = (
        np.abs(analytical - numerical)
        / np.maximum(
            1e-8,
            np.abs(analytical) + np.abs(numerical)
        )
    )

    return np.max(elementwise_error) # gradient의 모든 원소가 정확해야 하므로, 가장 오차를 큰 원소를 기준으로 검사 

# 이 검사 결과로 역전파 코드 어딘가에 버그가 있나?를 검증할 수 있다!

In [18]:
numerical_dW1 = numerical_gradient(W1)
numerical_db1 = numerical_gradient(b1)
numerical_dW2 = numerical_gradient(W2)
numerical_db2 = numerical_gradient(b2)

In [19]:
print(
    "W1:",
    max_relative_error(dW1, numerical_dW1)
)

print(
    "b1:",
    max_relative_error(db1, numerical_db1)
)

print(
    "W2:",
    max_relative_error(dW2, numerical_dW2)
)

print(
    "b2:",
    max_relative_error(db2, numerical_db2)
)

W1: 2.927391012246377e-07
b1: 1.0
W2: 1.246206533442286e-08
b2: 1.7264560356611982e-10


### b1의 실패 이유 분석하기

In [20]:
print("Z1:")
print(Z1)

print("\nanalytical db1:")
print(db1)

print("\nnumerical db1:")
print(numerical_db1)

Z1:
[[ 0.          0.          0.          0.          0.          0.
   0.          0.        ]
 [-0.00168012 -0.08530439  0.0879398   0.07777919  0.00660307  0.11272412
   0.04675093 -0.08592925]
 [ 0.03047171 -0.10399841  0.07504512  0.09405647 -0.19510352 -0.13021795
   0.01278404 -0.03162426]
 [ 0.02879159 -0.1893028   0.16298492  0.17183567 -0.18850045 -0.01749383
   0.05953497 -0.11755351]]

analytical db1:
[[ 7.05419839e-05  0.00000000e+00 -1.07837327e-02  6.12883483e-04
   2.30470860e-03  8.48925735e-03 -1.50077460e-02  0.00000000e+00]]

numerical db1:
[[ 0.00237523 -0.00599301 -0.00529342  0.00030085  0.00114932  0.00423345
  -0.00736686 -0.00096581]]


### ReLU에서의 문제점

내 생각에, ReLU의 미분 불가능한 지점에서 시도한게 문제같음

즉 0인데, b1 + h는 0 + 0.000001을 수행하기 때문에, 이 부분에서 문제가 발생한 것 같음

즉 ReLU의 꺾이는 지점을 가로질러 측정되었기에 문제가 발생한 것 같음

따라서 중앙 차분이 계산한 값과 analytical backward가 선택한 값이 다르고..

이 경우에는 역전파가 틀린 게 아니라, 미분이 정의되지 않은 지점에서 gradient check를 수행한 것이 문제 같음...

고등학교 수학때 배웠는데 ㅠㅠ

b1 - h → ReLU의 음수 영역

b1 + h → ReLU의 양수 영역

```
analytical db1[0,1] =  0
numerical  db1[0,1] = -0.00599301
```

analytical gradient는 Z1 > 0 조건을 사용한다.

반면 중앙 차분은 bias를 양쪽으로 움직인다.

그렇기에 양쪽 함수의 행동이 다르기 때문에 numerical gradient는 0이 아닌 값을 얻는다

### 해결하자!

1. Gradient Check를 할 때 ReLU의 꺾이는 지점을 피해야 한다(Seed 재설정)

2. Kink 트래킹(Tracking) 및 경고 스킵 처리를 한다

3. Smooth Activation Function 으로 역전파 검증 후 교체


일단, 시드를 재설정해서 이 문제가 맞는지부터 확인하자.

In [21]:
# ReLU의 0 지점을 피하도록 bias 변경
b1 = np.array([
    [0.03, -0.02, 0.01, -0.03,
     0.02, 0.04, -0.01, 0.05]
])

# 순전파 재계산
Z1 = X @ W1 + b1
A1 = np.maximum(0, Z1)

Z2 = A1 @ W2 + b2
Y_hat = 1 / (1 + np.exp(-Z2))

loss = -np.mean(
    y * np.log(Y_hat + epsilon)
    + (1 - y) * np.log(1 - Y_hat + epsilon)
)

# 역전파 재계산
N = X.shape[0]

dZ2 = (Y_hat - y) / N
dW2 = A1.T @ dZ2
db2 = np.sum(dZ2, axis=0, keepdims=True)

dA1 = dZ2 @ W2.T
dZ1 = dA1 * (Z1 > 0)

dW1 = X.T @ dZ1
db1 = np.sum(dZ1, axis=0, keepdims=True)

In [22]:
print("minimum |Z1|:", np.min(np.abs(Z1)))

numerical_db1 = numerical_gradient(b1)

print(
    "b1 maximum relative error:",
    max_relative_error(db1, numerical_db1)
)

minimum |Z1|: 0.0027840403167285373
b1 maximum relative error: 7.555002065155222e-07


이 결과로, b1의 역전파가 틀린 것이 아니라 ReLU가 미분되지 않는 $Z_1=0$에서 gradient check를 수행했기 때문에 발생한 거짓 실패임을 증명했다

즉, ReLU의 0 지점에서 가장 가까운 값은
$$min |Z_1|=0.002784$$
인데, 

중앙차분 간격은 
$$h = 0.00001$$
이기에 ReLU의 꺾이는 점에서 충분히 떨어져 있다.

### 전체 파라미터 최종검사

In [23]:
numerical_dW1 = numerical_gradient(W1)
numerical_db1 = numerical_gradient(b1)
numerical_dW2 = numerical_gradient(W2)
numerical_db2 = numerical_gradient(b2)

In [24]:
errors = {
    "W1": max_relative_error(dW1, numerical_dW1),
    "b1": max_relative_error(db1, numerical_db1),
    "W2": max_relative_error(dW2, numerical_dW2),
    "b2": max_relative_error(db2, numerical_db2),
}

for name, error in errors.items():
    status = "PASS" if error < 1e-5 else "FAIL"
    print(f"{name}: {error:.3e}  {status}")

W1: 4.827e-07  PASS
b1: 7.555e-07  PASS
W2: 1.575e-08  PASS
b2: 1.820e-09  PASS


## 실제로 오류를 내고, 디버깅해보자

```db2 = np.sum(dZ2, axis=0, keepdims=True)```가 맞지만, sum이 아니라 mean을 해보자

```buggy_db2 = np.mean(dZ2, axis=0, keepdims=True)```를 해보도록하자

$$\text{buggy }db_2
=
\frac{\text{correct }db_2}{N}$$

In [25]:
buggy_db2 = np.mean(
    dZ2,
    axis=0,
    keepdims=True
)

buggy_error = max_relative_error(
    buggy_db2,
    numerical_db2
)

print("correct db2:", db2)
print("buggy db2:", buggy_db2)
print("numerical db2:", numerical_db2)
print("buggy relative error:", buggy_error)
print(
    "status:",
    "PASS" if buggy_error < 1e-5 else "FAIL"
)

correct db2: [[0.00198407]]
buggy db2: [[0.00049602]]
numerical db2: [[0.00198407]]
buggy relative error: 0.6000000011650639
status: FAIL


### 결과 분석

예측했던 대로, 산술평균을 계산하였기에 buggy db2는 correct db2에 비해 1/4정도의 크기가 나왔다.

이러한 오류는 단순한 Shape 체크로는 발견할 수 없다.


상대 오차가 0.6인 이유는?

정상 gradient를 $g$라고 하면:
$$
g_n=g,\qquad g_{\text{buggy}}=\frac{g}{4}
$$
상대오차는:
$$
\frac{|g-g/4|}{|g|+|g/4|}
=
\frac{3g/4}{5g/4}
=
\frac35
=
0.6
$$


따라서 0.6....은 우연히 나온 값이 아니라, 정확히 이 버그의 크기를 반영한다.

### 세 번째 원인 - $h$를 잘못 고르면

지금까지 두 가지 실패 원인을 봤다.

1. **ReLU kink** : 역전파는 맞지만 미분 불가능한 지점에서 검사했다 (`b1: 1.0`)

2. **Reduction 불일치** : sum을 mean으로 바꿔 스케일이 어긋났다 (`0.6`)

세 번째는 $h$ 자체다.

왜 $h$가 문제가 될까?

미분의 정의는 극한이다.

$$
\frac{\partial L}{\partial w}
=
\lim_{h \to 0}
\frac{L(w+h)-L(w-h)}{2h}
$$

<br>

정의대로라면 $h$가 작을수록 참값에 가까워진다.

$h=10^{-14}$가 $h=10^{-5}$보다 정확해야 한다.

**하지만 이건 실수(real number) 위에서의 이야기다.**

컴퓨터는 실수를 유한한 자릿수로만 표현한다.

$h$를 너무 작게 만들면 `loss_plus`와 `loss_minus`가 거의 같은 값이 되고,

그 차이를 담을 자릿수가 남지 않는다.

즉 두 가지 오차가 서로 반대로 움직인다.

- $h$가 **크면** : 극한에서 멀어져 생기는 오차 (절단 오차, $O(h^2)$)

- $h$가 **작으면** : 두 loss의 차이가 표현 한계에 먹히는 오차 (자릿수 소실)

In [26]:
# h를 바꿔가며 같은 gradient check를 반복한다

# W2로 검사하는 이유:
# W2는 Z1에 영향을 주지 않으므로 ReLU kink 문제가 섞이지 않는다
# 즉 h의 영향만 따로 관찰할 수 있다

h_values = [
    1e-1, 1e-2, 1e-3, 1e-4, 1e-5,
    1e-6, 1e-7, 1e-8, 1e-10, 1e-12, 1e-14,
]

for h_test in h_values:
    numerical = numerical_gradient(W2, h=h_test)
    error = max_relative_error(dW2, numerical)
    status = "PASS" if error < 1e-5 else "FAIL"

    print(f"h = {h_test:.0e}   relative error = {error:.3e}   {status}")

h = 1e-01   relative error = 1.039e-05   FAIL
h = 1e-02   relative error = 1.039e-07   PASS
h = 1e-03   relative error = 9.889e-10   PASS
h = 1e-04   relative error = 3.510e-09   PASS
h = 1e-05   relative error = 1.575e-08   PASS
h = 1e-06   relative error = 1.843e-07   PASS
h = 1e-07   relative error = 1.629e-06   PASS
h = 1e-08   relative error = 3.533e-05   FAIL
h = 1e-10   relative error = 3.155e-03   FAIL
h = 1e-12   relative error = 1.818e-01   FAIL
h = 1e-14   relative error = 1.000e+00   FAIL


### 결과 분석 - U자 곡선

오차가 단조롭게 줄지 않고, 중간에서 최소가 된 뒤 다시 커진다.

서로 반대 방향으로 움직이는 두 오차가 겹쳐 있기 때문이다.

$$
\text{전체 오차}(h)
\;\approx\;
\underbrace{C_1 h^2}_{\text{절단 오차}}
\;+\;
\underbrace{C_2 \frac{\varepsilon}{h}}_{\text{자릿수 소실}}
$$

여기서 $\varepsilon$은 float64의 표현 한계로, 약 $2.2\times10^{-16}$이다.

### 실무에서는

이 문제에서 최소점은 $10^{-3}$이지만, 최적 $h$는 함수의 곡률에 따라 달라진다.
관례적으로 $10^{-4}\sim10^{-6}$을 쓰면 안전하다.

우리가 계속 써온 $h=10^{-5}$는 오차 $1.575\times10^{-8}$로,
"거의 완벽" 기준인 $10^{-7}$보다 충분히 작다.

**$h$를 작게 할수록 정확해질 것 같지만, 사실이 아니다.**

In [27]:
# h가 작아질 때 loss_plus와 loss_minus를 직접 들여다본다

target = np.unravel_index(
    np.argmax(np.abs(dW2)),
    dW2.shape
)
# gradient의 절댓값이 가장 큰 원소를 고른다
# np.unravel_index(): flat 인덱스를 (행, 열) 형태로 바꿔준다

saved = W2[target]

print("target index:", target)
print("analytical gradient:", dW2[target])
print()

for h_test in [1e-5, 1e-8, 1e-12, 1e-14]:
    W2[target] = saved + h_test
    loss_plus = compute_loss()

    W2[target] = saved - h_test
    loss_minus = compute_loss()

    W2[target] = saved

    difference = loss_plus - loss_minus

    print(f"h = {h_test:.0e}")
    print(f"  loss_plus  = {loss_plus:.17f}")
    print(f"  loss_minus = {loss_minus:.17f}")
    print(f"  difference = {difference:.3e}")
    print(f"  gradient   = {difference / (2 * h_test):.8f}")
    print()

target index: (np.int64(5), np.int64(0))
analytical gradient: -0.011223416430319513

h = 1e-05
  loss_plus  = 0.69402043093787680
  loss_minus = 0.69402065540620550
  difference = -2.245e-07
  gradient   = -0.01122342

h = 1e-08
  loss_plus  = 0.69402054305972749
  loss_minus = 0.69402054328419571
  difference = -2.245e-10
  gradient   = -0.01122341

h = 1e-12
  loss_plus  = 0.69402054317195039
  loss_minus = 0.69402054317197293
  difference = -2.254e-14
  gradient   = -0.01126876

h = 1e-14
  loss_plus  = 0.69402054317196149
  loss_minus = 0.69402054317196171
  difference = -2.220e-16
  gradient   = -0.01110223



### 자릿수 소실 (Catastrophic Cancellation)

출력을 보면 `loss_plus`와 `loss_minus`의 **앞자리가 계속 같다**.

```
h = 1e-05
  loss_plus  = 0.69402043093787680
  loss_minus = 0.69402065540620550
                ^^^^^^^ 7번째 자리부터 다름

h = 1e-12
  loss_plus  = 0.69402054317195039
  loss_minus = 0.69402054317197293
                           ^^^ 14번째 자리부터 다름
```

우리가 원하는 정보는 **뒷자리의 미세한 차이**인데,

$h$가 작아질수록 그 차이가 뒤로 밀린다.

float64는 유효숫자를 약 16자리까지만 표현한다.

loss가 $0.694$ 정도이므로 표현 가능한 최소 간격은 대략 $10^{-16}$이다.

$h=10^{-14}$에서는 두 loss의 차이가 $-2.22\times10^{-16}$,

즉 float64가 구분할 수 있는 최소 단위의 2배 수준이다.


이렇게 비슷한 두 수를 빼서 유효숫자를 잃는 현상을
**자릿수 소실(catastrophic cancellation)** 이라고 한다.

$h$를 줄이면 수식적으로는 더 정확해지지만,
컴퓨터의 표현 한계 때문에 실제로는 더 부정확해진다.

## 오차 원인 정리

지금까지 gradient check가 실패하는 경우를 세 가지 만났다.

| 원인 | 상대오차의 특징 | 실제 버그인가? |
|---|---|---|
| ReLU kink | 정확히 **1.0** (한쪽만 0) | 아니오 (검사 위치 문제) |
| Reduction 불일치 (sum ↔ mean) | 고정된 상수 (N=4일 때 **0.6**) | 예 |
| $h$ 선택 실패 | $h$를 바꾸면 값이 따라 변함 | 아니오 (수치 문제) |

### 진단 순서

역전파를 의심하기 전에 이 순서로 확인한다.

1. **상대오차가 정확히 1.0인가?**

    한쪽 gradient만 0이라는 뜻이다. ReLU kink이거나 죽은 뉴런이다.
    `min |Z1|`을 출력해 $h$보다 충분히 큰지 본다.

2. **$h$를 바꾸면 오차가 변하는가?**

    변하면 수치 문제다. $10^{-4}\sim10^{-6}$ 범위에서 다시 검사한다.
    안 변하면 진짜 역전파 버그다.

3. **오차가 특정 상수에 고정되어 있는가?**

    스케일이 어긋난 것이다. $N$으로 나누기 누락, sum과 mean 혼동,
    학습률을 gradient에 미리 곱한 경우 등을 확인한다.

### 결론

코드가 실행되고, loss가 감소하고, shape이 맞는다는 사실만으로는
역전파가 정확하다고 판단할 수 없다.

그리고 gradient check가 실패했다고 해서
역전파가 틀렸다고 판단할 수도 없다.

**오차의 크기와 패턴이 원인을 알려준다.**